# Walkthrough — Clasificador Grupo 3 / Ciberseguridad

Recorrido del pipeline en celdas separadas — pensado para tomar capturas paso a paso.

Requiere: tunnel SSH al VPS arriba (`make tunnel` desde `examen/`).

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().resolve().parents[1]  # examen/
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT.parent))             # examen/classifier importable como package
import lib  # carga .env
from lib.connections import minio
from classifier.src.config import BUCKET, GROUP_PREFIX
print(f'bucket: {BUCKET} · prefix: {GROUP_PREFIX}')
print('buckets en MinIO:')
for b in minio().list_buckets().get('Buckets', []):
    print(f'  • {b["Name"]}')

## Bronze — inspección rápida

In [ ]:
import io, polars as pl
s3 = minio()
buf = io.BytesIO()
s3.download_fileobj(BUCKET, f'{GROUP_PREFIX}/bronze/index.parquet', buf)
buf.seek(0)
bronze = pl.read_parquet(buf)
print(f'Total PDFs en Bronze: {bronze.height}')
print(f'Tamaño total: {bronze["size_bytes"].sum()/1024/1024:.1f} MB')
bronze.head(5).select(['code','original_filename','arxiv_id','topic_tag_local','year_from_arxiv','size_bytes'])

## Silver — metadata extraída

In [ ]:
buf = io.BytesIO()
s3.download_fileobj(BUCKET, f'{GROUP_PREFIX}/silver/silver.parquet', buf)
buf.seek(0)
silver = pl.read_parquet(buf)
print(f'Silver rows: {silver.height}')
silver.select(['code','year','title','score','decision']).head(10)

### Distribución de scores

In [ ]:
silver.group_by('score').agg(pl.len().alias('count')).sort('score', descending=True)

### Distribución de decisiones

In [ ]:
silver.group_by('decision').agg(pl.len().alias('count')).sort('count', descending=True)

## Gold — papers seleccionados

In [ ]:
buf = io.BytesIO()
s3.download_fileobj(BUCKET, f'{GROUP_PREFIX}/gold/gold.parquet', buf)
buf.seek(0)
gold = pl.read_parquet(buf)
print(f'Gold count: {gold.height}')
print(f'  score 5: {(gold["score"]==5).sum()}')
print(f'  score 4: {(gold["score"]==4).sum()}')
gold.select(['code','year','title','score','sbert_cosine']).head(20)

### Distribución por año (Gold)

In [ ]:
gold.group_by('year').agg(pl.len().alias('count')).sort('year', descending=True)

## Ranking final — Top 20

In [ ]:
ranking = (gold
    .sort(['score','sbert_cosine','year'], descending=[True, True, True])
    .with_row_index(name='ranking', offset=1)
    .select(['ranking','code','year','title','score','keywords_matched','decision'])
    .head(20))
ranking

## Justificación de un paper específico

In [ ]:
# Cambiá el código por el que quieras inspeccionar
code_to_inspect = gold['code'][0] if gold.height else 'PAPER_0001'
row = silver.filter(pl.col('code') == code_to_inspect).to_dicts()[0]
print(f"Código: {row['code']}")
print(f"Año:    {row['year']} ({row['year_confidence']})")
print(f"Score:  {row['score']}  ({row['decision']})")
print(f"Título: {row['title']}")
print()
print(f"Justificación: {row['justificacion']}")
print()
print(f"Keywords matched: {row['keywords_matched']}")
print()
print(f"Abstract ({len(row['abstract'] or '')} chars):")
print((row['abstract'] or '')[:600] + '…')